# 02 — Perbandingan Regresi

Lima algoritma regresi dibandingkan pada data, fitur, praproses, split, lipatan
CV, dan metrik yang sama:

**Linear Regression, Random Forest, Gradient Boosting, XGBoost, KNN.**

Tidak ada hyperparameter tuning: semua model memakai nilai bawaan (kecuali
`random_state` dan `n_jobs=1` agar hasil dapat direproduksi). Semua angka hasil
dihitung oleh kode di notebook ini.

In [1]:
import importlib.util
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import display

if importlib.util.find_spec("xgboost") is None:
    print("Memasang xgboost (diperlukan notebook model)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])

cwd = Path.cwd()
module_path = cwd / "perbandingan_common.py"
if not module_path.exists():
    docs_module = cwd / "docs" / "perbandingan_common.py"
    if docs_module.exists():
        sys.path.insert(0, str(docs_module.parent))
    else:
        url = "https://raw.githubusercontent.com/RobbyCatur/rumahpoint/main/docs/perbandingan_common.py"
        print("Mengunduh modul bersama dari GitHub ...")
        urllib.request.urlretrieve(url, module_path)
        sys.path.insert(0, str(cwd))
else:
    sys.path.insert(0, str(cwd))

import perbandingan_common as c

print("modul:", c.__file__)
print("repo terdeteksi:", c.REPO_ROOT)
print("direktori keluaran:", c.OUTPUT_DIR)
display(pd.DataFrame([c.describe_environment()]).T.rename(columns={0: "nilai"}))

modul: /opt/rumahpoint/docs/perbandingan_common.py
repo terdeteksi: /opt/rumahpoint
direktori keluaran: /opt/rumahpoint/docs/perbandingan_output


,nilai
python,3.12.3
platform,Linux-7.0.14-28-pve-x86_64-with-glibc2.39
sklearn,1.9.0
pandas,3.0.5
numpy,2.5.2
joblib,1.6.0
xgboost,3.4.1


## 1. Tiga rujukan dan mengapa perbandingan diulang

Angka di tabel ini berasal dari abstrak jurnal, bukan dari eksperimen kita.
Ketiganya memakai data kecil dan satu wilayah, dan mereka tidak sepakat tentang
model terbaik.

| Rujukan | Data | Model | Hasil yang dilaporkan | Pemenang |
| --- | --- | --- | --- | --- |
| Fitri (2023), JACOST 4(1), DOI 10.52158/jacost.v4i1.491 | 2.011 rumah, 6 atribut, 80/20 | Linear Regression, Random Forest, Gradient Boosted Trees | RF RMSE 0,440; LR 0,515; GBT 0,508; akurasi RF 81,5% | Random Forest |
| Sari & Prabowo (2025), Intellect 4(1), DOI 10.57255/intellect.v4i1.1385 | 1.642 entri Yogyakarta, harga Rp 7 juta–4,37 miliar | Random Forest, Gradient Boosting, XGBoost | XGBoost MSE 1,56 × 10¹⁴ Rp² dan R² 0,7746; unggul atas RF/GB | XGBoost |
| Hayya’ dkk. (2025), Sainteks 22(1), DOI 10.30595/sainteks.v22i1.25998 | 2.020 baris, 9 kolom fitur | KNN, Random Forest | RF MAE 54,80, MSE 7.004,70, R² 0,55; KNN MAE 87,64, MSE 13.314,95, R² 0,14 | Random Forest |

Pelajaran dari ketiganya: pemenang bergantung pada data dan metrik. Dataset
Rumah Point jauh lebih besar dan berasal dari banyak kota, sehingga hasilnya
perlu diukur ulang dengan protokol yang sama untuk semua kandidat.

## 2. Protokol yang sama untuk semua model

| Komponen | Keputusan |
| --- | --- |
| Data | CSV union setelah pembersihan notebook 01; baris latih/uji tetap |
| Split hold-out | 80% latih / 20% uji, `random_state=42` |
| CV | 5 lipatan `KFold(shuffle=True, random_state=42)` pada baris latih |
| Fitur | `luas_m2`, `luas_bangunan`, `jumlah_kamar`, `jumlah_kamar_mandi`, `lokasi` |
| Usia bangunan | tidak dipakai di perbandingan utama; diuji terpisah di bagian 6 |
| Praproses | imputasi median + `StandardScaler` untuk numerik; OneHot `lokasi`; fit di dalam pipeline per lipatan |
| Target | `log10(harga)` |
| Model | lima model, parameter bawaan, tanpa tuning |
| Metrik hold-out | MAE Rupiah, RMSE Rupiah, R² log10, R² Rupiah, MAPE |
| Metrik CV | rata-rata dan simpangan baku metrik yang sama |
| Waktu | waktu latih dan prediksi hold-out; rata-rata CV |
| Ukuran model | hasil `joblib.dump` tanpa kompresi |
| Aturan pemenang | rata-rata R² log10 CV tertinggi; jika seri, MAE Rupiah CV terendah; lalu R² hold-out |

Semua model menerima DataFrame yang sama dan pipeline praproses yang dibangun
oleh fungsi yang sama. KNN memakai seluruh baris latih (bukan subsampel) agar
data tetap identik.

In [2]:
df, rep, split, train_idx, test_idx, folds = c.load_bundle()
train_df, test_df = c.train_test_frames(df, train_idx, test_idx)
X_train = c.feature_frame(train_df)
X_test = c.feature_frame(test_df)
y_train = train_df[c.TARGET_REG]
y_test = test_df[c.TARGET_REG]
print(f"latih {len(X_train):,}; uji {len(X_test):,}")
print("fitur:", list(X_train.columns))
display(X_train.head(3))

=== SIDIK JARI DATA ===
berkas dataset        : /opt/rumahpoint/dataset/properti_multi_kota.csv
baris mentah          : 64,650
baris bersih          : 64,228
baris latih / uji     : 51,382 / 12,846
hash baris latih      : cfe3defcae087f44fa6a7189a319ef258f4138318a7532b315bf80a85dc9990e
hash baris uji        : 718f7f914fe83904e4899da30d2fc6b7a06e249431befe26985f93e43efe7aef
hash file dataset     : 34375406a6d72b21bad55e5784caddc931a85bb20c999414d3fdd6b2f5774d18
lipatan 0 (indeks uji): f298c1b2ee29f320d674c00a96af6e48ce0dff211e66a5d6798761cf3ae938d4
lipatan 1 (indeks uji): c177e4c57ee746c9d021815cecfbc4e8128469d08d1617f086033940cee7efe6
lipatan 2 (indeks uji): 7c6dfcd5824b61717f80eeddde5371bb4c93adb3a21beddf20741479178c025e
lipatan 3 (indeks uji): 9956fdc032bc252aeb7d0ecac5531626bac860dee609478a49110408ee8b19cd
lipatan 4 (indeks uji): a0b04506e601150298462828e0306e977fd83ad22063aa0ebd10d9663b7c51dc


latih 51,382; uji 12,846
fitur: ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi', 'lokasi']


,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,lokasi
0,83.0,60.0,2.0,2.0,Kabupaten Karimun
1,108.0,70.0,2.0,2.0,Kabupaten Sigi
2,66.0,40.0,2.0,1.0,Kabupaten Gresik


### Parameter tiap model

Tabel dihasilkan dari objek model yang benar-benar akan dilatih. Nilai yang
tidak dicetak berarti estimator tidak memakai parameter itu.

In [3]:
models = c.regression_models()
display(c.model_parameter_table(models))

,model,n_jobs,n_estimators,max_depth,random_state,learning_rate,eval_metric,n_neighbors,weights
0,Linear Regression,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Random Forest,1.0,100.0,NaN,42.0,NaN,NaN,NaN,NaN
2,Gradient Boosting,NaN,100.0,3.0,42.0,0.1,NaN,NaN,NaN
3,XGBoost,1.0,NaN,NaN,42.0,NaN,NaN,NaN,NaN
4,KNN,1.0,NaN,NaN,NaN,NaN,NaN,5.0,uniform


## 3. Hasil hold-out

Satu model dilatih pada seluruh baris latih, lalu diuji sekali pada baris uji.
Waktu prediksi mencakup satu panggilan `predict`. Ukuran model adalah ukuran file
joblib, bukan ukuran memori saat melayani.

Jika hasil tersimpan cocok dengan daftar model, jumlah baris latih/uji, dan
jumlah lipatan, notebook memakai ulang hasil itu; jika tidak, semua model
dihitung ulang dari awal. Pada eksekusi bersih, seluruh angka dihitung ulang.


In [4]:
rows = []
expected_models = list(models)
holdout_path = c.OUTPUT_DIR / "regression_results_holdout.csv"
if holdout_path.exists():
    cached = pd.read_csv(holdout_path)
    cache_valid = (
        cached["model"].astype(str).tolist() == expected_models
        and len(cached) == len(expected_models)
        and int(cached["n_train"].iloc[0]) == len(X_train)
        and int(cached["n_test"].iloc[0]) == len(X_test)
    )
else:
    cached = None
    cache_valid = False

if cache_valid:
    holdout = cached
    print("Memakai hasil hold-out tersimpan yang cocok dengan data dan daftar model:", holdout_path)
else:
    for name, factory in models.items():
        row, pred, _ = c.evaluate_holdout(
            name, factory, X_train, y_train, X_test, y_test, task="regresi"
        )
        rows.append(row)
    holdout = pd.DataFrame(rows)
    c.save_csv("regression_results_holdout.csv", holdout)

holdout_view = holdout[[
    "model", "mae_rupiah", "rmse_rupiah", "r2_log10", "r2_rupiah",
    "mape_pct", "fit_seconds", "predict_seconds", "model_size_mb",
]]
display(holdout_view.style.format({
    "mae_rupiah": "Rp {:,.0f}",
    "rmse_rupiah": "Rp {:,.0f}",
    "r2_log10": "{:.4f}",
    "r2_rupiah": "{:.4f}",
    "mape_pct": "{:.2f}%",
    "fit_seconds": "{:.2f}",
    "predict_seconds": "{:.3f}",
    "model_size_mb": "{:.2f}",
}))


Memakai hasil hold-out tersimpan yang cocok dengan data dan daftar model: /opt/rumahpoint/docs/perbandingan_output/regression_results_holdout.csv


,model,mae_rupiah,rmse_rupiah,r2_log10,r2_rupiah,mape_pct,fit_seconds,predict_seconds,model_size_mb
0,Linear Regression,"Rp 69,961,665,995","Rp 5,663,928,547,146",0.4634,-1607405.7889,470.14%,15.97,0.285,0.02
1,Random Forest,"Rp 581,266,918","Rp 2,974,447,350",0.8134,0.5567,83.88%,208.83,1.852,319.44
2,Gradient Boosting,"Rp 779,751,623","Rp 3,369,880,236",0.7217,0.4310,108.10%,138.44,0.719,0.15
3,XGBoost,"Rp 679,679,796","Rp 3,161,594,155",0.7887,0.4992,85.41%,14.31,0.304,0.33
4,KNN,"Rp 733,343,934","Rp 3,344,529,741",0.7500,0.4395,94.45%,4.37,25.290,172.50


## 4. Hasil validasi silang 5 lipatan

Setiap lipatan membangun pipeline baru dari nol. Metrik dihitung pada lipatan
uji, lalu dirata-ratakan. Simpangan baku menunjukkan stabilitas antar-lipatan.

In [5]:
fold_frames = []
summaries = []
expected_models = list(models)
cv_path = c.OUTPUT_DIR / "regression_results_cv.csv"
folds_path = c.OUTPUT_DIR / "regression_results_cv_folds.csv"
if cv_path.exists() and folds_path.exists():
    cached_cv = pd.read_csv(cv_path)
    cached_folds = pd.read_csv(folds_path)
    cache_valid = (
        cached_cv["model"].astype(str).tolist() == expected_models
        and len(cached_cv) == len(expected_models)
        and cached_folds["model"].nunique() == len(expected_models)
        and int(cached_cv["n_train"].iloc[0]) == len(X_train)
        and cached_folds.groupby("model").size().eq(len(folds)).all()
    )
else:
    cached_cv = None
    cached_folds = None
    cache_valid = False

if cache_valid:
    cv = cached_cv
    cv_folds = cached_folds
    print("Memakai hasil CV tersimpan yang cocok dengan data, lipatan, dan daftar model:", cv_path)
else:
    for name, factory in models.items():
        folds_df, summary = c.evaluate_cv(
            name, factory, X_train, y_train, folds, task="regresi"
        )
        folds_df.insert(0, "model", name)
        fold_frames.append(folds_df)
        summaries.append(summary)
    cv_folds = pd.concat(fold_frames, ignore_index=True)
    cv = pd.DataFrame(summaries)
    c.save_csv("regression_results_cv_folds.csv", cv_folds)
    c.save_csv("regression_results_cv.csv", cv)

cv_view = cv[[
    "model", "mae_rupiah_mean", "mae_rupiah_std", "rmse_rupiah_mean",
    "r2_log10_mean", "r2_log10_std", "r2_rupiah_mean", "mape_pct_mean",
    "fit_seconds_mean", "predict_seconds_mean",
]]
display(cv_view.style.format({
    "mae_rupiah_mean": "Rp {:,.0f}",
    "mae_rupiah_std": "Rp {:,.0f}",
    "rmse_rupiah_mean": "Rp {:,.0f}",
    "r2_log10_mean": "{:.4f}",
    "r2_log10_std": "{:.4f}",
    "r2_rupiah_mean": "{:.4f}",
    "mape_pct_mean": "{:.2f}%",
    "fit_seconds_mean": "{:.2f}",
    "predict_seconds_mean": "{:.3f}",
}))


Memakai hasil CV tersimpan yang cocok dengan data, lipatan, dan daftar model: /opt/rumahpoint/docs/perbandingan_output/regression_results_cv.csv


,model,mae_rupiah_mean,mae_rupiah_std,rmse_rupiah_mean,r2_log10_mean,r2_log10_std,r2_rupiah_mean,mape_pct_mean,fit_seconds_mean,predict_seconds_mean
0,Linear Regression,"Rp 99,516,982,830","Rp 68,703,917,636","Rp 7,448,616,087,358",0.4848,0.0116,-2671325.0037,449.68%,6.40,0.082
1,Random Forest,"Rp 631,521,340","Rp 15,836,991","Rp 3,440,092,344",0.8153,0.0071,0.5721,87.71%,204.45,1.948
2,Gradient Boosting,"Rp 832,958,586","Rp 16,235,802","Rp 4,104,706,037",0.7299,0.0085,0.3922,102.19%,99.61,0.563
3,XGBoost,"Rp 716,432,422","Rp 18,688,480","Rp 3,763,560,822",0.7958,0.0078,0.4904,93.01%,7.82,0.105
4,KNN,"Rp 795,848,652","Rp 19,500,318","Rp 4,067,141,661",0.7531,0.0091,0.4038,89.40%,3.83,15.669


## 5. Pemenang dan alasan

Aturan sudah ditetapkan sebelum melihat hasil: rata-rata R² log10 CV tertinggi,
tie-break MAE Rupiah CV, lalu R² hold-out. Tabel peringkat di bawah dihitung
dari hasil, bukan diketik manual.

In [6]:
combined = holdout.merge(cv, on="model", suffixes=("", "_cv"))
winner = c.choose_regression_winner(combined)
print("Pemenang:", winner["model"])
print("Alasan:", winner["reason"])
peringkat = pd.DataFrame({
    "peringkat": range(1, len(winner["ranking"]) + 1),
    "model": winner["ranking"],
})
display(peringkat)

Pemenang: Random Forest
Alasan: Rata-rata R2 log10 CV tertinggi (0.8153) dengan MAE CV Rp 631,521,340; hold-out R2 log10 0.8134 dan MAE Rp 581,266,918.


,peringkat,model
0,1,Random Forest
1,2,XGBoost
2,3,KNN
3,4,Gradient Boosting
4,5,Linear Regression


## 6. Uji bukti fitur usia bangunan

Usia bangunan tidak dipakai di perbandingan utama. Untuk membuktikan keputusan
itu, Random Forest default dilatih dua kali pada split dan lipatan yang sama:
tanpa usia dan dengan `usia_bangunan_norm`.

**Kriteria ditetapkan sebelum eksperimen:**

| Syarat | Ambang |
| --- | --- |
| Selisih R² log10 hold-out (dengan − tanpa) | minimal +0,010 |
| Selisih R² log10 CV (dengan − tanpa) | minimal +0,005 |
| MAE Rupiah hold-out | harus turun |

Jika ketiganya tidak terpenuhi, usia dibuang. Ambang ini disengaja konservatif
karena usia kosong pada sebagian besar baris dan tahun acuan tidak sama untuk
semua sumber.

In [7]:
age_test = c.run_age_proof(df, train_idx, test_idx, folds)
usia_rows = []
for label in ("tanpa_usia", "dengan_usia"):
    h = age_test["results"][label]["holdout"]
    v = age_test["results"][label]["cv"]
    usia_rows.append({
        "varian": label,
        "r2_log10_holdout": h["r2_log10"],
        "mae_rupiah_holdout": h["mae_rupiah"],
        "r2_log10_cv": v["r2_log10_mean"],
        "mae_rupiah_cv": v["mae_rupiah_mean"],
    })
display(pd.DataFrame(usia_rows).style.format({
    "r2_log10_holdout": "{:.4f}",
    "mae_rupiah_holdout": "Rp {:,.0f}",
    "r2_log10_cv": "{:.4f}",
    "mae_rupiah_cv": "Rp {:,.0f}",
}))
print("Delta:", age_test["delta"])
print("Keputusan:", age_test["keputusan"])
print("Alasan:", age_test["alasan"])

[hold-out] Random Forest (tanpa_usia): R2_log10=0.8134, MAE=Rp 581,266,918, latih=203.4s, prediksi=4.3s, ukuran=nan MB


[cv Random Forest (tanpa_usia)] lipatan 0: R2_log10=0.8128, MAE=Rp 658,624,034


[cv Random Forest (tanpa_usia)] lipatan 1: R2_log10=0.8185, MAE=Rp 628,013,317


[cv Random Forest (tanpa_usia)] lipatan 2: R2_log10=0.8241, MAE=Rp 635,342,754


[cv Random Forest (tanpa_usia)] lipatan 3: R2_log10=0.8031, MAE=Rp 610,172,450


[cv Random Forest (tanpa_usia)] lipatan 4: R2_log10=0.8178, MAE=Rp 625,454,146


... (7 outputs left out)

[cv] Random Forest (dengan_usia): R2_log10=0.8153 ± 0.0072, MAE=Rp 634,606,654


[usia] delta R2 hold-out=-0.0000, delta R2 CV=+0.0000, delta MAE hold-out=Rp +1,903,957 -> dibuang


,varian,r2_log10_holdout,mae_rupiah_holdout,r2_log10_cv,mae_rupiah_cv
0,tanpa_usia,0.8134,"Rp 581,266,918",0.8153,"Rp 631,521,340"
1,dengan_usia,0.8134,"Rp 583,170,875",0.8153,"Rp 634,606,654"


Delta: {'r2_holdout': -1.2055886796202842e-05, 'r2_cv': 4.127871396886196e-05, 'mae_rupiah_holdout': 1903957.2874720097, 'mae_rupiah_cv': 3085314.210693598}
Keputusan: dibuang
Alasan: Minimal satu ambang tidak terpenuhi; usia tidak dipakai sebagai fitur utama.


## 7. Simpan ringkasan

Ringkasan ini dibaca notebook kesimpulan. Angka yang disimpan adalah angka yang
baru saja dihitung; tidak ada angka yang diketik ulang.

In [8]:
summary = {
    "task": "regresi",
    "winner": winner,
    "protocol": {
        "n_rows": rep["clean_rows"],
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "split_seed": split["seed"],
        "test_size": split["test_size"],
        "cv_folds": len(folds),
        "features": c.MAIN_FEATURES,
        "target": c.TARGET_REG,
        "preprocessing": "median imputer + StandardScaler (numerik), OneHot (lokasi), fit per lipatan",
        "models": list(models),
        "winner_rule": "mean CV R2 log10 tertinggi; tie-break mean MAE Rupiah CV; lalu R2 hold-out",
    },
    "sidik_jari": {
        "raw_rows": rep["raw_rows"],
        "clean_rows": rep["clean_rows"],
        "train_sha256": split["train_sha256"],
        "test_sha256": split["test_sha256"],
    },
    "holdout": holdout.to_dict("records"),
    "cv": cv.to_dict("records"),
    "age_test": age_test,
    "jurnal": [
        {"nama": "Fitri (2023), JACOST 4(1)", "DOI": "10.52158/jacost.v4i1.491",
         "pemenang": "Random Forest"},
        {"nama": "Sari & Prabowo (2025), Intellect 4(1)", "DOI": "10.57255/intellect.v4i1.1385",
         "pemenang": "XGBoost"},
        {"nama": "Hayya' dkk. (2025), Sainteks 22(1)", "DOI": "10.30595/sainteks.v22i1.25998",
         "pemenang": "Random Forest"},
    ],
}
c.save_json("regression_summary.json", summary)
print("Berkas hasil regresi selesai ditulis.")

[tulis] /opt/rumahpoint/docs/perbandingan_output/regression_summary.json


Berkas hasil regresi selesai ditulis.


## Catatan membaca hasil

- R² log10 mengukur kecocokan pada skala target; R² Rupiah sangat dipengaruhi
  beberapa rumah bernilai ekstrem.
- MAPE bisa besar pada rumah murah; gunakan bersama MAE/RMSE, bukan sendirian.
- Waktu dan ukuran model bergantung pada mesin. Angka waktu bukan patokan
  produksi, tetapi perbandingan relatif pada mesin yang sama.
- Model produksi di `cloud_ml/model/` tidak diubah oleh notebook ini.